# Proyecto 1 - Obtención y Limpieza de Datos

## Universidad del Valle de Guatemala

### CC3084 - Data Science

---

## Objetivo

Automatizar la obtención de los establecimientos educativos del Ministerio de Educación de Guatemala correspondientes al nivel **Diversificado** para todos los departamentos del país.

Todo el proceso debe ser reproducible mediante código.

---

Autor: Osman Emanuel de León García
```

In [1]:
# Instalación de librerías

%pip install selenium webdriver-manager pandas openpyxl xlrd

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip available: 22.3 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os
import time
from pathlib import Path

import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

from selenium.common.exceptions import TimeoutException
from selenium.common.exceptions import NoSuchElementException

from webdriver_manager.chrome import ChromeDriverManager
from selenium.webdriver.chrome.service import Service

In [3]:
# -----------------------------
# Configuración del proyecto
# -----------------------------

from pathlib import Path

URL = "https://www.mineduc.gob.gt/BUSCAESTABLECIMIENTO_GE/wbfBuscar.aspx"

# Estructura de carpetas
DATA_FOLDER = Path("data")

RAW_FOLDER = DATA_FOLDER / "raw"
CSV_FOLDER = DATA_FOLDER / "csv"
CLEAN_FOLDER = DATA_FOLDER / "clean"

RAW_FOLDER.mkdir(parents=True, exist_ok=True)
CSV_FOLDER.mkdir(parents=True, exist_ok=True)
CLEAN_FOLDER.mkdir(parents=True, exist_ok=True)

DOWNLOAD_TIMEOUT = 60

In [4]:
departamentos = {
    "16": "Alta_Verapaz",
    "15": "Baja_Verapaz",
    "04": "Chimaltenango",
    "20": "Chiquimula",
    "00": "Ciudad_Capital",
    "02": "El_Progreso",
    "05": "Escuintla",
    "01": "Guatemala",
    "13": "Huehuetenango",
    "18": "Izabal",
    "21": "Jalapa",
    "22": "Jutiapa",
    "17": "Peten",
    "09": "Quetzaltenango",
    "14": "Quiche",
    "11": "Retalhuleu",
    "03": "Sacatepequez",
    "12": "San_Marcos",
    "06": "Santa_Rosa",
    "07": "Solola",
    "10": "Suchitepequez",
    "08": "Totonicapan",
    "19": "Zacapa"
}

In [5]:
def esperar_elemento(driver, by, value, timeout=20):
    """
    Espera hasta que un elemento esté presente.
    """
    return WebDriverWait(driver, timeout).until(
        EC.presence_of_element_located((by, value))
    )


def esperar_click(driver, by, value, timeout=20):
    """
    Espera hasta que un elemento sea clickeable.
    """
    return WebDriverWait(driver, timeout).until(
        EC.element_to_be_clickable((by, value))
    )


def esperar_descarga(carpeta, timeout=60):

    tiempo = time.time()

    while time.time() - tiempo < timeout:

        archivos_tmp = list(Path(carpeta).glob("*.crdownload"))

        if not archivos_tmp:

            archivo = Path(carpeta) / "establecimiento.xls"

            if archivo.exists():
                return archivo

        time.sleep(1)

    raise TimeoutException("La descarga tardó demasiado.")

In [6]:
def renombrar_excel(nombre_departamento):
    """
    Renombra el archivo descargado con el nombre del departamento.
    """

    archivo = RAW_FOLDER / "establecimiento.xls"

    nuevo = RAW_FOLDER / f"{nombre_departamento}.xls"

    if nuevo.exists():
        nuevo.unlink()

    archivo.rename(nuevo)

    return nuevo

In [7]:
options = webdriver.ChromeOptions()

prefs = {
    "download.default_directory": str(RAW_FOLDER.resolve()),
    "download.prompt_for_download": False,
    "download.directory_upgrade": True,
    "safebrowsing.enabled": True
}

options.add_experimental_option("prefs", prefs)

options.add_argument("--start-maximized")

driver = webdriver.Chrome(
    service=Service(ChromeDriverManager().install()),
    options=options
)

wait = WebDriverWait(driver,20)

driver.get(URL)

In [8]:
for codigo, nombre in departamentos.items():

    print("="*60)
    print(f"Descargando {nombre}")
    print("="*60)

    # Departamento

    Select(
        esperar_elemento(
            driver,
            By.ID,
            "_ctl0_ContentPlaceHolder1_cmbDepartamento"
        )
    ).select_by_value(codigo)

    # Nivel

    Select(
        esperar_elemento(
            driver,
            By.ID,
            "_ctl0_ContentPlaceHolder1_cmbNivel"
        )
    ).select_by_value("46")

    # Buscar

    esperar_click(
        driver,
        By.ID,
        "_ctl0_ContentPlaceHolder1_IbtnConsultar"
    ).click()

    # Esperar que aparezca el botón Exportar

    esperar_click(
        driver,
        By.ID,
        "_ctl0_ContentPlaceHolder1_btnExportar",
        timeout=30
    )

    # Exportar

    driver.find_element(
        By.ID,
        "_ctl0_ContentPlaceHolder1_btnExportar"
    ).click()

    # Esperar descarga

    esperar_descarga(RAW_FOLDER, DOWNLOAD_TIMEOUT)

    # Renombrar

    renombrar_excel(nombre)

    print(f"{nombre} descargado correctamente.\n")

Descargando Alta_Verapaz
Alta_Verapaz descargado correctamente.

Descargando Baja_Verapaz
Baja_Verapaz descargado correctamente.

Descargando Chimaltenango
Chimaltenango descargado correctamente.

Descargando Chiquimula
Chiquimula descargado correctamente.

Descargando Ciudad_Capital
Ciudad_Capital descargado correctamente.

Descargando El_Progreso
El_Progreso descargado correctamente.

Descargando Escuintla
Escuintla descargado correctamente.

Descargando Guatemala
Guatemala descargado correctamente.

Descargando Huehuetenango
Huehuetenango descargado correctamente.

Descargando Izabal
Izabal descargado correctamente.

Descargando Jalapa
Jalapa descargado correctamente.

Descargando Jutiapa
Jutiapa descargado correctamente.

Descargando Peten
Peten descargado correctamente.

Descargando Quetzaltenango
Quetzaltenango descargado correctamente.

Descargando Quiche
Quiche descargado correctamente.

Descargando Retalhuleu
Retalhuleu descargado correctamente.

Descargando Sacatepequez
Sacat

In [ ]:
print("="*50)
print("Descarga completada.")
print(f"Departamentos descargados: {len(departamentos)}")
print(f"Archivos guardados en: {RAW_FOLDER.resolve()}")
print("="*50)

Proceso finalizado.
